# Mini Project 3 — Real-Time Recommendation System

Walk-through notebook for the e-commerce / real-time-intelligence variant of the assignment.

**Pipeline**
1. Load Amazon Electronics reviews
2. Preprocess & split 80 / 20
3. Train ALS, evaluate RMSE, tune if > 1.5
4. (run in terminal) Kafka producer streams events
5. (run in terminal) Spark Structured Streaming computes windows + alerts + recs
6. Inspect the streaming outputs from disk

> The producer and the streaming job each need their own terminal, so this notebook focuses on the **batch** side and on **post-hoc inspection** of the streaming sinks.

## 0. Imports & paths

In [ ]:
import sys
from pathlib import Path

# Make `src/` importable
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import config
print('Project root:', PROJECT_ROOT)
print('Historical parquet target:', config.HISTORICAL_PARQUET)

## 1. Download & preprocess data

Run `python src/download_data.py` from a terminal **once** (the download is large). The cell below just verifies the output Parquet exists and peeks at it.

In [ ]:
import pandas as pd

assert config.HISTORICAL_PARQUET.exists(), 'Run python src/download_data.py first'
hist = pd.read_parquet(config.HISTORICAL_PARQUET)
print(f'{len(hist):,} historical rows')
print('unique users:', hist['als_user'].nunique())
print('unique items:', hist['als_item'].nunique())
hist.head()

In [ ]:
# rating distribution — sanity check
hist['rating'].value_counts().sort_index().plot(kind='bar', title='Rating distribution')

## 2. Train ALS (or load already-trained model)

If you have already run `python src/train_als.py`, load the saved model. Otherwise run the cell below — it imports the same function the script uses.

In [ ]:
import json

if not config.MODEL_DIR.exists():
    print('No model found — running training (this takes a few minutes)...')
    import train_als
    train_als.main()

log_path = config.MODEL_DIR.parent / 'tuning_log.json'
if log_path.exists():
    log = json.loads(log_path.read_text())
    print('Best RMSE :', log['best_rmse'])
    print('Best params:', log['best_params'])
    print('Trials    :', len(log['trials']))
    pd.DataFrame(log['trials'])

## 3. Inspect the streaming sinks

These cells assume the producer and the streaming job have been running for at least ~30 seconds. Run them in two separate terminals:

```bash
# terminal A
python src/spark_streaming.py
# terminal B
python src/kafka_producer.py --inject-spike
```

In [ ]:
def read_sink(p):
    files = list(Path(p).rglob('*.parquet'))
    if not files:
        return pd.DataFrame()
    return pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)

items = read_sink(config.WINDOW_METRICS_DIR / 'items')
users = read_sink(config.WINDOW_METRICS_DIR / 'users')
recs  = read_sink(config.RECOMMENDATIONS_DIR / 'data')
print('item windows :', len(items))
print('user windows :', len(users))
print('recommendations:', len(recs))

In [ ]:
# Latest trending items (live system view)
if not items.empty:
    latest = items[items['window_end'] == items['window_end'].max()]
    display(latest.sort_values('trending_score', ascending=False).head(10))

In [ ]:
# Latency check — bonus requirement < 5 s
if not recs.empty:
    print('mean latency :', recs['latency_seconds'].mean(), 's')
    print('p95 latency  :', recs['latency_seconds'].quantile(0.95), 's')

In [ ]:
# Alerts
import json as _json
def read_json_dir(p):
    rows = []
    for f in Path(p).rglob('*.json'):
        for line in f.read_text().splitlines():
            if line.strip():
                rows.append(_json.loads(line))
    return pd.DataFrame(rows)

alerts = pd.concat([read_json_dir(config.ALERTS_DIR / 'item'),
                    read_json_dir(config.ALERTS_DIR / 'user')], ignore_index=True)
alerts.sort_values('emitted_at', ascending=False).head(20)

## 4. Discussion points (see REPORT.md for the full write-up)

- **Partitioning** — 2 partitions keyed on `user_id` so per-user counters stay local.
- **Watermark** — 1 minute, balancing late-data tolerance against state growth.
- **Custom metric** — `trending_score = log1p(count) * avg_rating / 5` damps viral spikes while still rewarding genuine popularity.
- **Hybrid recs** — `0.7 · als_score + 0.3 · trending_score` re-rank.
- **Late data** — events older than `watermark` are dropped from windowed aggregations but still logged in the raw Kafka topic if you need to re-process.